# Smart Lost & Found - train the custom YOLO11 model

Run this notebook on **Google Colab** with a GPU (`Runtime > Change runtime type > T4 GPU`).

It downloads the Roboflow *Lost and Found* dataset (rnomo, v2, CC BY 4.0), removes the `None` class, re-splits the images 70 / 20 / 10, trains `yolo11n`, evaluates on the test split and downloads `best.pt`.

Dataset: https://universe.roboflow.com/rnomo/lost-and-found-nlb4x-e6ayk/dataset/2

## 1. Install packages and check the GPU

In [ ]:
%pip install -q ultralytics roboflow

import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU - training will be very slow. Enable one under Runtime > Change runtime type.')

## 2. Download the dataset (YOLOv8 format)

**Option A (recommended):** add your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar) and give this notebook access. The key is never written into the notebook.

**Option B:** download the dataset zip from Roboflow in *YOLOv8* format and upload it when prompted.

In [ ]:
import os
import zipfile
from pathlib import Path

RAW_DIR = Path('datasets/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get('ROBOFLOW_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        api_key = None

if api_key:
    from roboflow import Roboflow
    project = Roboflow(api_key=api_key).workspace('rnomo').project('lost-and-found-nlb4x-e6ayk')
    project.version(2).download('yolov8', location=str(RAW_DIR))
else:
    from google.colab import files
    print('No API key found - upload the YOLOv8-format zip from Roboflow.')
    uploaded = files.upload()
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        archive.extractall(RAW_DIR)

print('Raw dataset in', RAW_DIR.resolve())

## 3. Clean and re-split

Same logic as `scripts/prepare_dataset.py` (kept inline so the notebook runs on Colab without the rest of the project): read the class names from `data.yaml`, drop `None`, remap the IDs to `0 Bottle, 1 Umbrella, 2 Calculator, 3 Charger, 4 Key`, merge everything and re-split 70 / 20 / 10 with seed 42.

In [ ]:
import random
import shutil

import yaml

CLASSES = ['Bottle', 'Umbrella', 'Calculator', 'Charger', 'Key']
DROP_CLASS = 'None'
RATIOS = {'train': 0.70, 'valid': 0.20, 'test': 0.10}
SEED = 42
OUT_DIR = Path('datasets/lost_and_found')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Class names come from the dataset itself - never assume the IDs.
source_yaml = sorted(RAW_DIR.rglob('data.yaml'), key=lambda p: len(p.parts))[0]
names = yaml.safe_load(source_yaml.read_text())['names']
source_names = [str(names[k]) for k in sorted(names, key=int)] if isinstance(names, dict) else [str(n) for n in names]
print('Original classes:', source_names)

new_ids = {name.lower(): i for i, name in enumerate(CLASSES)}
id_map = {}
for old_id, name in enumerate(source_names):
    key = name.strip().lower()
    if key in new_ids:
        id_map[old_id] = new_ids[key]
    elif key != DROP_CLASS.lower():
        raise ValueError(f'Unexpected class {name!r} in the dataset')
assert len(id_map) == len(CLASSES), 'A required class is missing from the dataset'

pairs = [(img, img.parent.parent / 'labels' / f'{img.stem}.txt')
         for img in sorted(RAW_DIR.rglob('*'))
         if img.suffix.lower() in IMAGE_EXTENSIONS and img.parent.name == 'images']
random.Random(SEED).shuffle(pairs)
n_train = int(len(pairs) * RATIOS['train'])
n_valid = int(len(pairs) * RATIOS['valid'])
splits = {'train': pairs[:n_train],
          'valid': pairs[n_train:n_train + n_valid],
          'test': pairs[n_train + n_valid:]}

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
instances = {name: 0 for name in CLASSES}
used = set()
for split, items in splits.items():
    (OUT_DIR / split / 'images').mkdir(parents=True)
    (OUT_DIR / split / 'labels').mkdir(parents=True)
    for img, label in items:
        stem, n = img.stem, 1
        while stem in used:
            stem, n = f'{img.stem}_{n}', n + 1
        used.add(stem)
        kept = []
        for line in (label.read_text().splitlines() if label.exists() else []):
            parts = line.split()
            if parts and int(float(parts[0])) in id_map:
                new_id = id_map[int(float(parts[0]))]
                kept.append(' '.join([str(new_id), *parts[1:]]))
                instances[CLASSES[new_id]] += 1
        shutil.copy2(img, OUT_DIR / split / 'images' / f'{stem}{img.suffix.lower()}')
        (OUT_DIR / split / 'labels' / f'{stem}.txt').write_text('\n'.join(kept) + ('\n' if kept else ''))

DATA_YAML = OUT_DIR / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(OUT_DIR.resolve()),
    'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
    'nc': len(CLASSES), 'names': CLASSES}, sort_keys=False))

print('Images per split:', {split: len(items) for split, items in splits.items()})
print('Instances per class:', instances)

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(data=str(DATA_YAML), epochs=50, imgsz=640, batch=16, patience=10, seed=42)
RUN_DIR = Path(results.save_dir)
print('Run folder:', RUN_DIR)

## 5. Evaluate on the test split

The test images were never seen during training or early stopping, so these are the numbers to quote in the report.

In [ ]:
best = YOLO(str(RUN_DIR / 'weights' / 'best.pt'))
metrics = best.val(data=str(DATA_YAML), split='test', plots=True)
EVAL_DIR = Path(metrics.save_dir)

print(f"{'class':<12}{'precision':>10}{'recall':>10}{'mAP50':>10}{'mAP50-95':>10}")
print(f"{'all':<12}{metrics.box.mp:>10.3f}{metrics.box.mr:>10.3f}{metrics.box.map50:>10.3f}{metrics.box.map:>10.3f}")
for i, class_id in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f'{metrics.names[int(class_id)]:<12}{p:>10.3f}{r:>10.3f}{ap50:>10.3f}{ap:>10.3f}')

In [ ]:
# Confusion matrix, PR curve and sample predictions (screenshot these for the report).
from IPython.display import Image as ShowImage, display

for pattern in ['confusion_matrix.png', '*PR_curve.png', 'val_batch0_pred.jpg']:
    for plot in sorted(EVAL_DIR.glob(pattern)):
        print(plot.name)
        display(ShowImage(filename=str(plot), width=800))

## 6. Download `best.pt`

Save the downloaded file as `models/best.pt` in the project folder.

In [ ]:
from google.colab import files

files.download(str(RUN_DIR / 'weights' / 'best.pt'))